# Week 5: Data Visualization and Communication
**Prepared by:** Tushar Sharma

Story: what does this retail transaction dataset tell us about spend distribution, category totals, quantity, discounts, time, and payment methods? Each chart is paired with a careful interpretation.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(Path("../data/retail_store_sales_raw.csv"))
for col in ["Price Per Unit", "Quantity", "Total Spent"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")
print("Shape:", df.shape)
display(df.head())

## 1. How are transaction spend values distributed?

In [ ]:
df["Total Spent"].dropna().hist(bins=30, figsize=(8, 4.5))
plt.title("Distribution of Retail Transaction Spend")
plt.xlabel("Total Spent"); plt.ylabel("Transaction count")
plt.tight_layout(); plt.show()
print("Mean:", df["Total Spent"].mean(), "| Median:", df["Total Spent"].median())

**Interpretation:** Compare mean and median to understand skew. Missing totals are excluded from the histogram.

## 2. Which categories have the highest recorded spend?

In [ ]:
category = df.groupby("Category")["Total Spent"].sum(min_count=1).sort_values()
category.plot(kind="barh", figsize=(9, 5))
plt.title("Recorded Spend by Product Category")
plt.xlabel("Total Spent"); plt.ylabel("Category")
plt.tight_layout(); plt.show()
display(category.sort_values(ascending=False).to_frame("Recorded Total Spent"))

**Interpretation:** Category totals are recorded sales value, not profit. Missing totals and different transaction counts affect the comparison.

## 3. How does quantity relate to transaction spend?

In [ ]:
plt.figure(figsize=(7.5, 4.8))
plt.scatter(df["Quantity"], df["Total Spent"], alpha=0.25)
plt.title("Quantity and Transaction Spend")
plt.xlabel("Quantity"); plt.ylabel("Total Spent")
plt.tight_layout(); plt.show()

**Interpretation:** Quantity and spend are expected to relate because total is tied to unit price and quantity. The association is partly mathematical by construction.

## 4. Is recorded spend different by discount status?

In [ ]:
df["Discount Applied"] = df["Discount Applied"].astype("string")
groups = [df.loc[df["Discount Applied"] == "False", "Total Spent"].dropna(),
          df.loc[df["Discount Applied"] == "True", "Total Spent"].dropna()]
plt.figure(figsize=(7.5, 4.8))
plt.boxplot(groups, tick_labels=["No discount", "Discount applied"], showfliers=False)
plt.title("Spend by Recorded Discount Status"); plt.ylabel("Total Spent")
plt.tight_layout(); plt.show()

**Interpretation:** Compare medians and spreads, but exclude unknown discount statuses and avoid claiming causation.

## 5. How does recorded spend change over time?

In [ ]:
dated = df.dropna(subset=["Transaction Date", "Total Spent"]).copy()
dated["Month"] = dated["Transaction Date"].dt.to_period("M").astype(str)
monthly = dated.groupby("Month")["Total Spent"].sum()
monthly.plot(figsize=(10, 4.8))
plt.title("Monthly Recorded Retail Spend")
plt.xlabel("Month"); plt.ylabel("Total Spent")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

**Interpretation:** The last available date is in January 2025, so the final month/year may be incomplete. Do not compare partial and full periods without adjustment.

## 6. Which payment methods are most common?

In [ ]:
counts = df["Payment Method"].value_counts()
counts.plot(kind="bar", figsize=(7.5, 4.8))
plt.title("Transaction Count by Payment Method")
plt.xlabel("Payment Method"); plt.ylabel("Number of transactions")
plt.xticks(rotation=20); plt.tight_layout(); plt.show()
display(counts.to_frame("Transaction count"))

**Interpretation:** These are transaction counts, not unique customer counts. Use clear titles and labels and explain what each chart can and cannot establish.